In [27]:
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


In [28]:
# Load the original training and testing datasets
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Training compounds:", len(train))
print("Testing compounds:", len(test))

Training compounds: 1349
Testing compounds: 338


In [29]:
# Create Morgan fingerprint generator
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=1024
)

# Function to convert SMILES into fingerprint arrays
def smiles_to_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    fingerprint = generator.GetFingerprint(mol)

    array = np.zeros(1024, dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fingerprint, array)

    return array


# Generate fingerprints for training and testing data
train_fps = [smiles_to_fingerprint(s) for s in train["smiles"]]
test_fps = [smiles_to_fingerprint(s) for s in test["smiles"]]

# Check whether any SMILES could not be processed
print("Invalid training SMILES:", sum(fp is None for fp in train_fps))
print("Invalid testing SMILES:", sum(fp is None for fp in test_fps))

Invalid training SMILES: 0
Invalid testing SMILES: 0


In [30]:
# Convert fingerprints to numerical arrays
train_fp_array = np.array(train_fps)
test_fp_array = np.array(test_fps)

# Original molecular descriptors
features = [
    "molecular_weight",
    "logp",
    "hba",
    "hbd",
    "psa",
    "rtb",
    "ro5_violations",
    "aromatic_rings",
    "heavy_atoms"
]

# Combine descriptors and fingerprints
X_train_new = np.hstack([
    train[features].to_numpy(),
    train_fp_array
])

X_test_new = np.hstack([
    test[features].to_numpy(),
    test_fp_array
])

y_train = train["pf_pEC50"]
y_test = test["pf_pEC50"]

print("Training features:", X_train_new.shape)
print("Testing features:", X_test_new.shape)

Training features: (1349, 1033)
Testing features: (338, 1033)


In [31]:
model_new = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    model_new,
    X_train_new,
    y_train,
    cv=cv,
    scoring={
        "r2": "r2",
        "rmse": "neg_root_mean_squared_error"
    }
)

print("FINGERPRINT MODEL CROSS-VALIDATION")
print("----------------------------------")
print(f"Mean R²: {cv_results['test_r2'].mean():.3f}")
print(f"R² SD: {cv_results['test_r2'].std():.3f}")
print(f"Mean RMSE: {-cv_results['test_rmse'].mean():.3f}")
print(f"RMSE SD: {cv_results['test_rmse'].std():.3f}")

FINGERPRINT MODEL CROSS-VALIDATION
----------------------------------
Mean R²: -45.257
R² SD: 18.023
Mean RMSE: 3.620
RMSE SD: 0.612


In [32]:
# Create Ridge Regression pipeline
ridge_model = make_pipeline(
    StandardScaler(),
    Ridge()
)

# Define regularisation strengths to test
param_grid = {
    "ridge__alpha": [0.1, 1, 10, 100, 1000]
}

# Same cross-validation settings as before
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Tune using training data only
ridge_search = GridSearchCV(
    ridge_model,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

ridge_search.fit(X_train_new, y_train)

print("RIDGE REGRESSION RESULTS")
print("------------------------")
print("Best alpha:", ridge_search.best_params_["ridge__alpha"])
print(f"Best CV RMSE: {-ridge_search.best_score_:.3f}")

RIDGE REGRESSION RESULTS
------------------------
Best alpha: 1000
Best CV RMSE: 0.531


In [33]:
# Test additional regularisation strengths
param_grid = {
    "ridge__alpha": [100, 300, 1000, 3000, 10000]
}

ridge_search = GridSearchCV(
    ridge_model,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

ridge_search.fit(X_train_new, y_train)

# Display results for every alpha
ridge_results = pd.DataFrame({
    "Alpha": ridge_search.cv_results_["param_ridge__alpha"],
    "Mean CV RMSE": -ridge_search.cv_results_["mean_test_score"],
    "RMSE SD": ridge_search.cv_results_["std_test_score"]
})

print(ridge_results.round(3))
print("\nBest alpha:", ridge_search.best_params_["ridge__alpha"])

   Alpha  Mean CV RMSE  RMSE SD
0    100         0.657    0.030
1    300         0.576    0.033
2   1000         0.531    0.039
3   3000         0.520    0.042
4  10000         0.528    0.043

Best alpha: 3000


In [34]:
# Original nine molecular descriptors
X_train_original = train[features]

# Cross-validation folds
outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

inner_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=21
)

# Model 1: Original Linear Regression
linear_model = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

linear_scores = cross_validate(
    linear_model,
    X_train_original,
    y_train,
    cv=outer_cv,
    scoring={
        "r2": "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error"
    }
)

# Model 2: Ridge Regression with fingerprints
ridge_pipeline = make_pipeline(
    StandardScaler(),
    Ridge()
)

# Alpha selection happens inside each outer training fold
ridge_search_nested = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid={
        "ridge__alpha": [100, 300, 1000, 3000, 10000]
    },
    cv=inner_cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

ridge_scores = cross_validate(
    ridge_search_nested,
    X_train_new,
    y_train,
    cv=outer_cv,
    scoring={
        "r2": "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error"
    },
    return_estimator=True
)

# Create comparison table
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression (9 descriptors)",
        "Ridge Regression (descriptors + fingerprints)"
    ],
    "Mean R²": [
        linear_scores["test_r2"].mean(),
        ridge_scores["test_r2"].mean()
    ],
    "R² SD": [
        linear_scores["test_r2"].std(),
        ridge_scores["test_r2"].std()
    ],
    "Mean RMSE": [
        -linear_scores["test_rmse"].mean(),
        -ridge_scores["test_rmse"].mean()
    ],
    "RMSE SD": [
        linear_scores["test_rmse"].std(),
        ridge_scores["test_rmse"].std()
    ],
    "Mean MAE": [
        -linear_scores["test_mae"].mean(),
        -ridge_scores["test_mae"].mean()
    ]
})

print("NESTED CROSS-VALIDATION COMPARISON")
print("----------------------------------")
print(comparison.round(3).to_string(index=False))

# Show alpha selected in each outer fold
selected_alphas = [
    fitted_model.best_params_["ridge__alpha"]
    for fitted_model in ridge_scores["estimator"]
]

print("\nSelected Ridge alpha in each outer fold:")
print(selected_alphas)


NESTED CROSS-VALIDATION COMPARISON
----------------------------------
                                        Model  Mean R²  R² SD  Mean RMSE  RMSE SD  Mean MAE
            Linear Regression (9 descriptors)    0.022  0.032      0.545    0.047     0.390
Ridge Regression (descriptors + fingerprints)    0.109  0.047      0.520    0.042     0.373

Selected Ridge alpha in each outer fold:
[3000, 3000, 3000, 3000, 3000]


In [35]:
# Create final Ridge model using selected alpha
final_ridge = make_pipeline(
    StandardScaler(),
    Ridge(alpha=3000)
)

# Train using all training compounds
final_ridge.fit(X_train_new, y_train)

# Predict pEC50 for test compounds
ridge_test_predictions = final_ridge.predict(X_test_new)

# Calculate test performance
ridge_test_r2 = r2_score(y_test, ridge_test_predictions)

ridge_test_rmse = np.sqrt(
    mean_squared_error(y_test, ridge_test_predictions)
)

ridge_test_mae = mean_absolute_error(
    y_test, ridge_test_predictions
)

# Display final results
print("FINAL RIDGE REGRESSION TEST RESULTS")
print("-----------------------------------")
print(f"R²:   {ridge_test_r2:.3f}")
print(f"RMSE: {ridge_test_rmse:.3f}")
print(f"MAE:  {ridge_test_mae:.3f}")

FINAL RIDGE REGRESSION TEST RESULTS
-----------------------------------
R²:   0.076
RMSE: 0.560
MAE:  0.405


In [36]:
# Select the original nine molecular descriptors
features = [
    "molecular_weight",
    "logp",
    "hba",
    "hbd",
    "psa",
    "rtb",
    "ro5_violations",
    "aromatic_rings",
    "heavy_atoms"
]

# Use the original training dataset
X_train_9 = train[features]
y_train_9 = train["pf_pEC50"]

print("Training features:", X_train_9.shape)

Training features: (1349, 9)


In [37]:
# Create Ridge Regression pipeline
ridge_9 = make_pipeline(
    StandardScaler(),
    Ridge()
)

# Define alpha values
param_grid_9 = {
    "ridge__alpha": [0.1, 1, 10, 100, 300, 1000, 3000, 10000]
}

# Five-fold cross-validation
cv_9 = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Tune Ridge using training data only
ridge_9_search = GridSearchCV(
    ridge_9,
    param_grid_9,
    cv=cv_9,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

ridge_9_search.fit(X_train_9, y_train_9)

print("RIDGE REGRESSION - 9 DESCRIPTORS")
print("--------------------------------")
print("Best alpha:", ridge_9_search.best_params_["ridge__alpha"])
print(f"Best CV RMSE: {-ridge_9_search.best_score_:.3f}")

RIDGE REGRESSION - 9 DESCRIPTORS
--------------------------------
Best alpha: 100
Best CV RMSE: 0.545


In [38]:
# Compare alpha values
ridge_9_results = pd.DataFrame({
    "Alpha": ridge_9_search.cv_results_["param_ridge__alpha"],
    "Mean CV RMSE": -ridge_9_search.cv_results_["mean_test_score"],
    "RMSE SD": ridge_9_search.cv_results_["std_test_score"]
})

print(ridge_9_results.round(3))

     Alpha  Mean CV RMSE  RMSE SD
0      0.1         0.545    0.047
1      1.0         0.545    0.047
2     10.0         0.545    0.047
3    100.0         0.545    0.047
4    300.0         0.545    0.047
5   1000.0         0.546    0.046
6   3000.0         0.547    0.045
7  10000.0         0.550    0.043


In [39]:
# Ridge Regression with 9 descriptors:
# Nested cross-validation using training data only

ridge_9_nested = GridSearchCV(
    estimator=make_pipeline(
        StandardScaler(),
        Ridge()
    ),
    param_grid={
        "ridge__alpha": [0.1, 1, 10, 100, 300, 1000, 3000, 10000]
    },
    cv=inner_cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

ridge_9_scores = cross_validate(
    ridge_9_nested,
    X_train_9,
    y_train_9,
    cv=outer_cv,
    scoring={
        "r2": "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error"
    },
    return_estimator=True
)

print("RIDGE (9 DESCRIPTORS): NESTED CV")
print("--------------------------------")
print(f"Mean R²: {ridge_9_scores['test_r2'].mean():.3f}")
print(f"Mean RMSE: {-ridge_9_scores['test_rmse'].mean():.3f}")
print(f"Mean MAE: {-ridge_9_scores['test_mae'].mean():.3f}")

print("Selected alphas:", [
    estimator.best_params_["ridge__alpha"]
    for estimator in ridge_9_scores["estimator"]
])

RIDGE (9 DESCRIPTORS): NESTED CV
--------------------------------
Mean R²: 0.024
Mean RMSE: 0.545
Mean MAE: 0.390
Selected alphas: [100, 100, 100, 100, 100]


thus is all
